# Введение в MapReduce модель на Python


In [5]:
from typing import NamedTuple
from typing import Iterator

In [6]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [7]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [8]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [9]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [10]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [11]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [12]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [13]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [14]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [15]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [16]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [17]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [18]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [19]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.1882851527975156)),
 (1, np.float64(2.1882851527975156)),
 (2, np.float64(2.1882851527975156)),
 (3, np.float64(2.1882851527975156)),
 (4, np.float64(2.1882851527975156))]

## Inverted index

In [20]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('it', ['0', '1', '2']),
 ('is', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [21]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [22]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [23]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2)]),
 (1, [('banana', 2), ('is', 18), ('it', 18), ('what', 10)])]

## TeraSort

In [24]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.049378925097981696)),
   (None, np.float64(0.08572365196215037)),
   (None, np.float64(0.10330711362482303)),
   (None, np.float64(0.10545020137186334)),
   (None, np.float64(0.13105456964083662)),
   (None, np.float64(0.14458480941781138)),
   (None, np.float64(0.1542075559859014)),
   (None, np.float64(0.19712074179115224)),
   (None, np.float64(0.2654261664548643)),
   (None, np.float64(0.2823636731916299)),
   (None, np.float64(0.29695358934422156)),
   (None, np.float64(0.29936437863394627)),
   (None, np.float64(0.309909024747163)),
   (None, np.float64(0.3250078313088517)),
   (None, np.float64(0.33621620652780226)),
   (None, np.float64(0.4058366537993441)),
   (None, np.float64(0.4162140739711766)),
   (None, np.float64(0.41849260759989526)),
   (None, np.float64(0.48253127468650525)),
   (None, np.float64(0.4947098247657915))]),
 (1,
  [(None, np.float64(0.6144825990343921)),
   (None, np.float64(0.6542130667509847)),
   (None, np.float64(0.6803600

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [25]:
import random
input_list = [random.randint(0, 100) for _ in range(100)]


def RECORDREADER():
  for value in input_list:
    yield (0, value)


def MAP(id:int, value:int):
  yield (id, value)


def REDUCE(id:int, values:list):
  yield (id, max(values))


output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, 100)]

### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [26]:
inputL=[1,3,4,8]

def RECORDREADER():
  for value in inputL:
    yield(0,value)

def MAP(key: int, value: int):
    yield (key, (value, 1))

def REDUCE(key: int, array: list):
  total_sum = 0
  total_count = 0
  for val, count in array:
    total_sum += val
    total_count += count
  if total_count > 0:
    yield(key, total_sum / total_count)
  else:
    yield(key, 0)


output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, 4.0)]

### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [27]:
class User(NamedTuple):
  id: int
  age: int
  gender: str
  social_contacts: int

data = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def RECORDREADER(key: str):
  for user in data:
    yield (getattr(user, key), user)

def groupbykey_sorted(iterable):
  items = sorted(iterable, key=lambda kv: kv[0]) 
  out = []
  for k, v in items:
    if out and out[-1][0] == k:
      out[-1][1].append(v)
    else:
      out.append((k, [v]))
  return out

print("GroupBy age:")
print(groupbykey_sorted(RECORDREADER("age")), end="\n\n")

print("GroupBy gender:")
print(groupbykey_sorted(RECORDREADER("gender")), end="\n\n")

GroupBy age:
[(25, [User(id=1, age=25, gender='female', social_contacts=240), User(id=2, age=25, gender='female', social_contacts=500)]), (33, [User(id=3, age=33, gender='female', social_contacts=800)]), (55, [User(id=0, age=55, gender='male', social_contacts=20)])]

GroupBy gender:
[('female', [User(id=1, age=25, gender='female', social_contacts=240), User(id=2, age=25, gender='female', social_contacts=500), User(id=3, age=33, gender='female', social_contacts=800)]), ('male', [User(id=0, age=55, gender='male', social_contacts=20)])]



### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [28]:
data = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
     User(id=2, age=25, gender='female', social_contacts=240),
    User(id=3, age=33, gender='female', social_contacts=800),
    User(id=3, age=33, gender='female', social_contacts=800)
]

maps = 2
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for user in split:
      yield (user.id, user)

  split_size = int(np.ceil(len(data)/maps))
  for i in range(0, len(data), split_size):
    yield RECORDREADER(data[i:i+split_size])


def MAP(userId:int, user:User):
  yield (userId, user)

def REDUCE(_, usersList:list):
  yield (list(set(usersList)))

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

5 key-value pairs were sent over a network.


[(0,
  [[User(id=0, age=55, gender='male', social_contacts=20)],
   [User(id=2, age=25, gender='female', social_contacts=240)]]),
 (1,
  [[User(id=1, age=25, gender='female', social_contacts=240)],
   [User(id=3, age=33, gender='female', social_contacts=800)]])]

#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [29]:
data_list = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=240),
    User(id=3, age=33, gender='female', social_contacts=800)
]


def RECORDREADER():
  for user in data_list:
    yield (user.id, user)


def MAP(userId:int, user:User):
  if user.age >25:
    yield (user, user)


def REDUCE(userId:int, user:User):
  yield user


output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[[User(id=0, age=55, gender='male', social_contacts=20)],
 [User(id=3, age=33, gender='female', social_contacts=800)]]

### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [30]:
class UserProjection(NamedTuple):
  id: int
  social_contacts: int


data_list = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=240),
    User(id=3, age=33, gender='female', social_contacts=800)
]


def RECORDREADER():
  for user in data_list:
    yield (user.id, user)


def MAP(userId:int, user:User):
  new_user = UserProjection(userId, user.social_contacts)
  yield(new_user, new_user)


def REDUCE(key:UserProjection, usersList:list):
  yield (key, key)


output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(UserProjection(id=0, social_contacts=20),
  UserProjection(id=0, social_contacts=20)),
 (UserProjection(id=1, social_contacts=240),
  UserProjection(id=1, social_contacts=240)),
 (UserProjection(id=2, social_contacts=240),
  UserProjection(id=2, social_contacts=240)),
 (UserProjection(id=3, social_contacts=800),
  UserProjection(id=3, social_contacts=800))]

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [31]:
class Person(NamedTuple):
  id: int
  name: str

R = [
  Person(1, "Alice"),
  Person(2, "Bob"),
  Person(3, "Charlie")
]

S = [
  Person(2, "Bob"),
  Person(4, "David")
]

def RECORDREADER():
  for t in R:
    yield (None, t)
  for t in S:
    yield (None, t)

def MAP(PersonID:int, person:Person):
  yield (person, person)

def REDUCE(key:Person, personL:list):
  yield (key, key)


result = MapReduce(RECORDREADER, MAP, REDUCE)
result = list(result)
result

[(Person(id=1, name='Alice'), Person(id=1, name='Alice')),
 (Person(id=2, name='Bob'), Person(id=2, name='Bob')),
 (Person(id=3, name='Charlie'), Person(id=3, name='Charlie')),
 (Person(id=4, name='David'), Person(id=4, name='David'))]

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [32]:
class Person(NamedTuple):
  id: int
  name: str

R = [
  Person(1, "Alice"),
  Person(2, "Bob"),
  Person(3, "Charlie")
]

S = [
  Person(2, "Bob"),
  Person(3, "Charlie"),
  Person(4, "David")
]

def RECORDREADER():
  for t in R:
    yield (None, t)
  for t in S:
    yield (None, t)

def MAP(PersonID:int, person:Person):
  yield (person, person)

def REDUCE(key:Person, personL:list):
  if len(personL) == 2:
    yield key


result = MapReduce(RECORDREADER, MAP, REDUCE)
result = list(result)
result

[Person(id=2, name='Bob'), Person(id=3, name='Charlie')]

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [33]:
class Person(NamedTuple):
  id: int
  name: str

R = [
  Person(1, "Alice"),
  Person(2, "Bob"),
  Person(3, "Charlie"),
  Person(4, "David"),
  Person(5, "David")
]

S = [
  Person(2, "Bob"),
  Person(4, "David")
]

data_list = [R, S]

def RECORDREADER():
  for setId, lst in enumerate(data_list):
    for t in lst:
      yield (setId, t)

def MAP(setId: int, t: Person):
  yield (t, setId)

def REDUCE(t: Person, setList: Iterator[int]):
  setList = list(setList)
  if len(setList) == 1 and setList[0] == 0:
    yield (t, t)


output = list(MapReduce(RECORDREADER, MAP, REDUCE))
output

[(Person(id=1, name='Alice'), Person(id=1, name='Alice')),
 (Person(id=3, name='Charlie'), Person(id=3, name='Charlie')),
 (Person(id=5, name='David'), Person(id=5, name='David'))]

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [34]:
class Users(NamedTuple):
  id: int
  gender: str

class Salary(NamedTuple):
  gender: str
  salary: int

R = [
    Users(id=0, gender='male'),
    Users(id=1, gender='female'),
    Users(id=2, gender='female'),
]

S = [
    Salary(gender='male', salary=400),
    Salary(gender='female', salary=350)
]

def RECORDREADER():
  for t in R:
    yield (0, t)
  for t in S:
    yield (1, t)

def MAP(listId: int, t):
  if listId == 0:
    b = t.gender
    a = t.id
    yield (b, ("R", a))
  else:
    b = t.gender
    c = t.salary
    yield (b, ("S", c))

def REDUCE(b, values):
  As = [x for (tag, x) in values if tag == "R"]
  Cs = [x for (tag, x) in values if tag == "S"]
  for a in As:
    for c in Cs:
      yield (a, b, c)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
output

[(0, 'male', 400), (1, 'female', 350), (2, 'female', 350)]

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [35]:
class Salary(NamedTuple):
  groupId: int
  gender: str
  salary: int

data_1= [
    Salary(groupId=0, gender='female', salary=450),
    Salary(groupId=0, gender='female', salary=350),
    Salary(groupId=0, gender='female', salary=260),
    Salary(groupId=0, gender='female', salary=160),
    Salary(groupId=0, gender='female', salary=500),
]

data_2 = [
    Salary(groupId=1, gender='male', salary=580),
    Salary(groupId=1, gender='male', salary=485),
    Salary(groupId=1, gender='male', salary=300),
    Salary(groupId=1, gender='male', salary=400),
    Salary(groupId=1, gender='male', salary=200),
]

data = [data_1, data_2]

def RECORDREADER():
  for lists in data:
    for line in lists:
      yield (line.groupId, line)

def MAP(groupId:int, line:Salary):
  yield (groupId, line.salary)

def REDUCE(listId:int, data_salary:list):
  yield (listId, sum(data_salary)/len(data_salary))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, 344.0), (1, 393.0)]

#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [36]:
matrix = np.ones((3, 3))
vector = np.array([1, 0, 0, 0])

maps = 2
reducers = 1

def INPUTFORMAT():
    global maps

    def RECORDREADER(split):
        for i in range(split.shape[0]):
            for j in range(split.shape[1]):
                yield ((i, j), (split[i, j], vector[j], split.shape[1]))

    split_size = int(np.ceil(len(matrix) / maps))

    for i in range(0, len(matrix), split_size):
        yield RECORDREADER(matrix[i: i + split_size])


def MAP(coordinates: [int, int], values: [int, int, int]):
    i, j = coordinates
    matrix_value, vector_value, cols = values
    yield ((i, cols), matrix_value * vector_value)


def REDUCE(keys: [int, int], products: Iterator[NamedTuple]):
    i, cols = keys
    yield (i, sum(products) / (len(products) / cols))


partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output


9 key-value pairs were sent over a network.


[(0, [(0, np.float64(1.0)), (1, np.float64(1.0))])]

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [37]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [38]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J)  # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(small_mat.shape[0]):
   yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
  (i, k) = key
  s = 0.0
  for vw in values:
    s += vw
  yield ((i, k), s)

Проверьте своё решение

In [39]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [40]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [41]:
I = 2
J = 3
K = 4*10
M = np.random.rand(I, J)
N = np.random.rand(J, K)

def RECORDREADER():
  for i in range(M.shape[0]):
    for j in range(M.shape[1]):
      yield (None, ("M", i, j, M[i, j]))
  for j in range(N.shape[0]):
    for k in range(N.shape[1]):
      yield (None, ("N", j, k, N[j, k]))

def MAP(_, rec):
  tag = rec[0]
  if tag == "M":
    _, i, j, v = rec
    yield (j, ("M", i, v))
  else:
    _, j, k, w = rec
    yield (j, ("N", k, w))

def REDUCE(j, values):
  Ms = []
  Ns = []
  for tag, a, b in values:
    if tag == "M":
      Ms.append((a, b))
    else:
      Ns.append((a, b))
  for i, v in Ms:
    for k, w in Ns:
      yield ((i, k), v * w)

def MAP2(key, partial):
  yield (key, partial)

def REDUCE2(key, partials):
  s = 0.0
  for x in partials:
    s += x
  yield (key, s)

partials = list(MapReduce(RECORDREADER, MAP, REDUCE))
def RECORDREADER2():
  for (ik, vw) in partials:
    yield (ik, vw)

solution = list(MapReduce(RECORDREADER2, MAP2, REDUCE2))

# check
def asmatrix(reduce_output):
  I_ = max(i for ((i,k), _) in reduce_output) + 1
  K_ = max(k for ((i,k), _) in reduce_output) + 1
  mat = np.empty((I_, K_))
  for ((i,k), val) in reduce_output:
    mat[i, k] = val
  return mat

reference_solution = np.matmul(M, N)
print(np.allclose(reference_solution, asmatrix(solution)))

True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [42]:
def RECORDREADER_SMALL():
  for i in range(small_mat.shape[0]):
    for j in range(small_mat.shape[1]):
      yield (("R", i, j), float(small_mat[i, j]))

def RECORDREADER_BIG():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield (("S", j, k), float(big_mat[j, k]))

def INPUTFORMAT_1():

  R_items = list(RECORDREADER_SMALL())
  S_items = list(RECORDREADER_BIG())

  def split(items, parts):
    size = int(np.ceil(len(items) / parts)) if parts > 0 else len(items)
    for p in range(0, len(items), size):
      chunk = items[p:p+size]
      def rr(chunk=chunk):
        for x in chunk:
          yield x
      yield rr()

  m1 = max(1, maps // 2)
  m2 = max(1, maps - m1)
  yield from split(R_items, m1)
  yield from split(S_items, m2)

def MAP_1(k1, v1):
  tag = k1[0]
  if tag == "R":
    _, i, j = k1
    yield (j, ("R", i, v1))
  else:
    _, j, k = k1
    yield (j, ("S", k, v1))

def REDUCE_1(j, values: Iterator):
  Rs = []
  Ss = []
  for tag, a, b in values:
    if tag == "R":
      Rs.append((a, b))
    else:
      Ss.append((a, b))

  for i, mij in Rs:
    for k, njk in Ss:
      yield ((i, k), mij * njk)

step1_outputs = MapReduceDistributed(INPUTFORMAT_1, MAP_1, REDUCE_1)
partials = []
for _rid, it in step1_outputs:
  partials.extend(list(it))

def INPUTFORMAT_2():
  size = int(np.ceil(len(partials) / maps)) if maps > 0 else len(partials)
  for p in range(0, len(partials), size):
    chunk = partials[p:p+size]
    def rr(chunk=chunk):
      for x in chunk:
        yield x
    yield rr()

def MAP_2(k1, v1):
  yield (k1, v1)

def REDUCE_2(key, values: Iterator[float]):
  s = 0.0
  for x in values:
    s += x
  yield (key, s)

step2_outputs = MapReduceDistributed(INPUTFORMAT_2, MAP_2, REDUCE_2)
solution = []
for _rid, it in step2_outputs:
  solution.extend(list(it))

# Check
reference_solution = np.matmul(small_mat, big_mat)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I_ = max(i for ((i,k), vw) in reduce_output)+1
  K_ = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I_, K_))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

print(np.allclose(reference_solution, asmatrix(solution)))  # should return True

126 key-value pairs were sent over a network.
240 key-value pairs were sent over a network.
True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [44]:
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I, J)
big_mat   = np.random.rand(J, K)

reducers = 2

def make_recordreaders_small(num_readers: int):
  coords = [(i, j) for i in range(small_mat.shape[0]) for j in range(small_mat.shape[1])]
  chunks = [[] for _ in range(num_readers)]
  for idx, (i, j) in enumerate(coords):
    chunks[idx % num_readers].append((i, j))
  for chunk in chunks:
    def rr(chunk=chunk):
      for (i, j) in chunk:
        yield (("R", i, j), float(small_mat[i, j]))
    yield rr()

def make_recordreaders_big(num_readers: int):
  coords = [(j, k) for j in range(big_mat.shape[0]) for k in range(big_mat.shape[1])]
  chunks = [[] for _ in range(num_readers)]
  for idx, (j, k) in enumerate(coords):
    chunks[idx % num_readers].append((j, k))
  for chunk in chunks:
    def rr(chunk=chunk):
      for (j, k) in chunk:
        yield (("S", j, k), float(big_mat[j, k]))
    yield rr()

readers_small = 4
readers_big   = 5

def INPUTFORMAT_1():
  yield from make_recordreaders_small(readers_small)
  yield from make_recordreaders_big(readers_big)

def MAP_1(k1, v1):
  tag = k1[0]
  if tag == "R":
    _, i, j = k1
    yield (j, ("R", i, v1))
  else:
    _, j, k = k1
    yield (j, ("S", k, v1))

def REDUCE_1(j, values: Iterator):
  Rs = []
  Ss = []
  for tag, a, b in values:
    if tag == "R":
      Rs.append((a, b))
    else:
      Ss.append((a, b))
  for i, mij in Rs:
    for k, njk in Ss:
      yield ((i, k), mij * njk)

step1_outputs = MapReduceDistributed(INPUTFORMAT_1, MAP_1, REDUCE_1)
partials = []
for _rid, it in step1_outputs:
  partials.extend(list(it))

maps2 = 6

def INPUTFORMAT_2():
  size = int(np.ceil(len(partials) / maps2)) if maps2 > 0 else len(partials)
  for p in range(0, len(partials), size):
    chunk = partials[p:p+size]
    def rr(chunk=chunk):
      for x in chunk:
        yield x
    yield rr()

def MAP_2(k1, v1):
  yield (k1, v1)

def REDUCE_2(key, values: Iterator[float]):
  s = 0.0
  for x in values:
    s += x
  yield (key, s)

step2_outputs = MapReduceDistributed(INPUTFORMAT_2, MAP_2, REDUCE_2)
solution = []
for _rid, it in step2_outputs:
  solution.extend(list(it))

reference_solution = np.matmul(small_mat, big_mat)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I_ = max(i for ((i,k), _) in reduce_output) + 1
  K_ = max(k for ((i,k), _) in reduce_output) + 1
  mat = np.empty((I_, K_))
  for ((i,k), val) in reduce_output:
    mat[i, k] = val
  return mat

print("Correct (allclose):", np.allclose(reference_solution, asmatrix(solution)))  # should be True

126 key-value pairs were sent over a network.
240 key-value pairs were sent over a network.
Correct (allclose): True


Решение будет работать корректно только в том случае, если несколько RECORDREADER-ов вместе генерируют все элементы матрицы и каждый элемент встречается ровно один раз. Если же элементы будут пропущены или появятся несколько раз, результат умножения исказится. Пропуски будут эквивалентны нулям, а дубли приводят к повторному учёту вклада элемента в решение.